In [ ]:
import pandas as pd
import numpy as np

# Carrega as 3 planilhas da Equipe 06
df_contas = pd.read_csv("contas_clientes.csv")
df_devices = pd.read_csv("dispositivos_autenticados.csv")
df_tx = pd.read_csv("transacoes_financeiras.csv")

# ETAPA 1.1: Checar logins íntegros vs violados 
CONFIG_BASE = 0xC4  
df_devices["flag_int"] = df_devices["flag_seguranca_hex"].apply(int, base=16)
df_devices["nibble_sessao"] = df_devices["flag_int"] & 0x0F
CONFIG_COMPLIANCE = CONFIG_BASE & 0x0F

df_devices["divergencia_xor"] = df_devices["nibble_sessao"] ^ CONFIG_COMPLIANCE
df_devices["status_auditoria"] = np.where(df_devices["divergencia_xor"] == 0, "Íntegro", "Acesso Violado")

print("=== RESULTADO 1.1 ===")
print(df_devices["status_auditoria"].value_counts())

# ETAPA 1.2: Fazenda de dispositivos 
uso_devices = df_tx.groupby("id_device")["id_conta_origem"].agg(
    qtd_contas="nunique",
    contas_vinculadas=set
)
fazenda_devices = uso_devices[uso_devices["qtd_contas"] > 1].sort_values(by="qtd_contas", ascending=False)

print("\n=== RESULTADO 1.2 ===")
print(f"Dispositivos suspeitos: {len(fazenda_devices)}")
display(fazenda_devices.head(10))

=== RESULTADO 1.1 ===
status_auditoria
Acesso Violado    88
Íntegro            7
Name: count, dtype: int64

=== RESULTADO 1.2 ===
Dispositivos suspeitos: 95


,qtd_contas,contas_vinculadas
id_device,,
DEV_533,23,"{CTA_1054, CTA_1098, CTA_8806, CTA_1082, CTA_1..."
DEV_551,22,"{CTA_1008, CTA_8802, CTA_1076, CTA_1020, CTA_8..."
DEV_557,22,"{CTA_1054, CTA_1004, CTA_1069, CTA_1058, CTA_8..."
DEV_570,22,"{CTA_1052, CTA_1093, CTA_1059, CTA_1013, CTA_1..."
DEV_503,21,"{CTA_1049, CTA_1016, CTA_1099, CTA_1004, CTA_1..."
DEV_513,21,"{CTA_8802, CTA_1071, CTA_1083, CTA_1045, CTA_1..."
DEV_546,21,"{CTA_1006, CTA_1023, CTA_1099, CTA_1022, CTA_1..."
DEV_567,20,"{CTA_1098, CTA_1023, CTA_8806, CTA_1063, CTA_8..."
DEV_504,20,"{CTA_1082, CTA_1066, CTA_1052, CTA_1093, CTA_1..."
